# MBAI 448 | Week 7 Assignment: Retrieval Augmented Generation (RAG)

## Assignment Overview

This assignment has three acts:
1. **Act 1: Understand the problem and context** — Set up your working environment and document your understanding in an `agents.md` file.
2. **Act 2: Prototype a solution with AI technology** — Build a RAG system that combines vector search with an LLM to answer research questions, documenting your approach in a `README.md` file.
3. **Act 3: Socialize the solution with stakeholders** — Roleplay conversations with business stakeholders using GitHub Copilot to anticipate real-world concerns.

---

## Assignment Tools

This assignment assumes you will be working with GitHub Copilot in VS Code or Google Colab, and will require you to submit your chat history along with this notebook. If you are curious about how to work effectively with GitHub Copilot, please consult the [VS Code documentation](https://code.visualstudio.com/docs/copilot/overview).

Submissions that demonstrate thoughtless interaction with Copilot (e.g., asking Copilot to just read the notebook and produce all the outputs) will receive reduced credit.

## Business Goal / Case Statement

You are a new hire on the Research Enablement team at B10, a pharmaceutical informatics subsidiary of MultiCorp, Inc. Your team supports researchers working across different domains, who typically have separate "silos" of information. Your boss believes these teams might benefit from being able to access each other's information, and wants to make it easy for them to do so. They have curated an initial set of FAQs and want you to develop a prototype of an intelligent system that uses that resource to answer questions.

**Relevant Industry:** Pharmaceutical R&D

**Data Location:** `./data/chemlit_qa_full_dataset_1025_entries_trimmed.csv`

**Dataset:** [ChemLit-QA](https://openreview.net/forum?id=6PoHVQeeHU)

---

##### AI/ML Task(s)

- Question-Answering
- Information Retrieval

##### Algorithmic Technique(s)

- [Retrieval Augmented Generation (RAG)](https://huggingface.co/docs/transformers/en/model_doc/rag)
- [Text Embeddings](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html)
- [Vector Search (FAISS)](https://github.com/facebookresearch/faiss)

---

### Act 1: Understand the problem and context

### Step 0: Scope the work in agents.md

Before writing any code, create an `agents.md` file in your working directory. This file documents your working relationship with AI coding assistants like GitHub Copilot.

Your `agents.md` should include:

1. **Task Overview**: A brief description of what you're trying to accomplish—building a RAG system that retrieves relevant context from a FAQ dataset and generates natural language answers.

2. **Your Role**: What decisions you'll make, what you'll validate, and where your judgment is essential (e.g., embedding model selection, similarity thresholds, prompt design).

3. **AI Assistant's Role**: What you expect Copilot to help with (e.g., FAISS index setup, embedding pipeline code, LLM integration).

4. **Validation Strategy**: How you'll verify that AI-generated code works correctly (e.g., testing retrieval quality, comparing RAG outputs to direct LLM outputs).

5. **Quality Standards**: What "good enough" looks like for this prototype (e.g., retrieves relevant context, answers are grounded in source material, system knows when to decline).

---

### Act 2: Prototype a solution with AI technology

### Prototyping a RAG System for Research Knowledge Access

In this act, you will prototype a Retrieval Augmented Generation (RAG) system that combines vector search with a language model to answer research questions. The goal is to understand how grounding LLM outputs in retrieved context can improve reliability and transparency.

Throughout this act, use GitHub Copilot as a development assistant, following a disciplined loop in every step:

- **Plan**: Have Copilot draft a clear, plain-language plan describing what needs to happen and in what order.
- **Validate**: Review and refine that plan to ensure it does exactly what the step requires—no more, no less.
- **Execute**: Have Copilot implement the validated plan in code.
- **Check**: Perform one or two concrete actions that confirm the code worked and that you understand the result.

This is exploratory prototyping. The goal is to remain in contact with the system's real behavior at all times.

---

#### Environment Setup

If running locally in VS Code, you may want to create and activate a Python virtual environment.

##### On MacOS/Linux:
```
python -m venv venv
source venv/bin/activate
```

##### On Windows:
```
python -m venv venv
venv\Scripts\activate
```

### Step 1: Load and Explore Your Data

**Plan**: Load the ChemLit-QA dataset and explore its structure to understand what information is available. Sample a few entries.

**Validate**: You should load the data and be able to work with it. The HuggingFace datasets library is a good choice. 

**Execute**:

In [ ]:
# Environment Setup: Install required packages
# Run this cell once to install dependencies
!pip install -q pandas sentence-transformers faiss-cpu transformers torch

In [ ]:
# Step 1: Load and Explore the ChemLit-QA Dataset
import pandas as pd

# Load the dataset from the local CSV file
df = pd.read_csv("./data/chemlit_qa_full_dataset_1025_entries_trimmed.csv")

# Basic structure
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")

# Preview a few rows
print(f"\n--- Sample Entries ---")
for i in range(3):
    print(f"\n[Entry {i}]")
    print(f"  Question: {df.iloc[i]['Question'][:150]}...")
    print(f"  Answer:   {df.iloc[i]['Answer'][:150]}...")
    print(f"  Cluster:  {df.iloc[i]['Cluster_labels']}")

# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

# Distribution of cluster labels
print(f"\nCluster label distribution:\n{df['Cluster_labels'].value_counts()}")

**Check**: 
- Review the dataset structure. What columns are available? How many rows?
- Review several samples. What domains do the questions cover? How comprehensive are the answers?

---

### Step 2: Create and Examine Text Embeddings

**Plan**: Create a component to represent input questions as vector embeddings. 

**Validate**: Make sure you are creating an embedding, not just tokenizing, and that you are creating an embedding for the entire input rather than just a single word therein.

**Execute**:

In [ ]:
# Step 2: Create and Examine Text Embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a sentence-transformer model for creating text embeddings
# all-MiniLM-L6-v2 is a good balance of speed and quality for semantic search
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings for a few sample questions to test
sample_questions = [
    "What is the role of copper acetate in MOF film growth?",
    "How does UV-ozone treatment affect substrate surfaces?",
    "What is the weather like today?"  # out-of-domain test
]

sample_embeddings = embedding_model.encode(sample_questions)

print(f"Number of embeddings: {len(sample_embeddings)}")
print(f"Embedding dimension:  {sample_embeddings.shape[1]}")
print(f"Embedding dtype:      {sample_embeddings.dtype}")
print(f"\nFirst embedding (first 10 values): {sample_embeddings[0][:10]}")

**Check**: Examine embedding dimensions. Why are embeddings the same size regardless of input length?

---

In [ ]:
# Check: Examine embedding dimensions
# Embeddings are the same size regardless of input length because the model
# pools (mean-pools) all token-level representations into a single fixed-size vector.
# This is by design: a fixed dimensionality allows comparison via cosine similarity.

short_text = embedding_model.encode(["Hi"])
long_text = embedding_model.encode(["This is a much longer sentence about pharmaceutical research and chemical synthesis methods."])

print(f"Short input embedding shape: {short_text.shape}")
print(f"Long input embedding shape:  {long_text.shape}")
print(f"\nBoth produce {short_text.shape[1]}-dimensional vectors regardless of input length.")
print("This is because the transformer encoder processes all tokens, then mean-pools")
print("the token embeddings into a single fixed-size sentence representation.")

**🍎 Food for thought**: How are these different from Week 2's image embeddings? Are they the same as Week 2's text embeddings?

### Step 3: Set up Similarity Search

**Plan**: Use the component you built to create embeddings for every question in the dataset. Then create a mechanism for finding questions in the dataset similar to the user's query. 

**Validate**: This should produce a list of results from the dataset  ostensibly similar to the input.

**Execute**:

In [ ]:
# Step 3: Set up Similarity Search with FAISS
import faiss

# Create embeddings for all questions in the dataset
questions = df["Question"].tolist()
question_embeddings = embedding_model.encode(questions, show_progress_bar=True)

# Normalize embeddings for cosine similarity (FAISS IndexFlatIP on normalized vectors = cosine sim)
faiss.normalize_L2(question_embeddings)

# Build a FAISS index
embedding_dim = question_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # Inner product on normalized vectors = cosine similarity
index.add(question_embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}")

def search_faq(query, k=5):
    """Search the FAQ index for questions similar to the query."""
    query_embedding = embedding_model.encode([query])
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "question": df.iloc[idx]["Question"],
            "answer": df.iloc[idx]["Answer"],
        })
    return results

print("search_faq() function ready.")

**Check**: Test with several questions. Are the retrieved questions  relevant?

In [ ]:
# Check: Test similarity search with several queries
test_queries = [
    "How are MOF thin films synthesized?",
    "What solvents are used in nanoparticle preparation?",
    "What is the capital of France?"  # out-of-domain
]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    results = search_faq(query, k=3)
    for i, r in enumerate(results):
        print(f"\n  Result {i+1} (score: {r['score']:.4f}):")
        print(f"    Q: {r['question'][:120]}...")
        print(f"    A: {r['answer'][:120]}...")

**🍎 Food for thought**: Would this work if you used a different embedding model than you used for the input embeddings? What if the embeddings were the same size?


### Step 4: Load LLM to Communicate Answers

**Plan**: Generate natural language responses to input questions, using a large language model in concert with the component above to provide it grounding from the dataset. Feel free to use github-copilot-sdk or load a small model from HuggingFace, e.g., SmolLM2-1.7B-Instruct.

**Validate**: The LLM should generate fluent responses to the input question based on the retrieved content.

**Execute**:

In [ ]:
# Step 4: Load LLM to Communicate Answers
# Using SmolLM2-1.7B-Instruct — a small, fast model suitable for prototyping
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Create a text generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)

def rag_answer(query, k=3):
    """Retrieve relevant FAQ context and generate an answer using the LLM."""
    # Retrieve relevant context
    results = search_faq(query, k=k)
    
    # Build context from retrieved results
    context_parts = []
    for i, r in enumerate(results):
        context_parts.append(f"Q: {r['question']}\nA: {r['answer']}")
    context = "\n\n".join(context_parts)
    
    # Construct the prompt
    prompt = f"""You are a helpful research assistant. Answer the user's question based ONLY on the provided context. If the context does not contain enough information, say so.

Context:
{context}

User question: {query}

Answer:"""
    
    # Generate response
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=256, do_sample=False)
    response = output[0]["generated_text"][-1]["content"]
    
    return response, results

print(f"LLM loaded: {model_name}")
print("rag_answer() function ready.")

**Check**: Test this RAG pipeline. How well does it respond the input questions?

In [ ]:
# Check: Test the RAG pipeline
test_queries = [
    "How are MOF thin films synthesized?",
    "What methods are used to detect PFAS contamination?",
]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    answer, _ = rag_answer(query)
    print(f"\nRAG Answer:\n{answer}")

### Step 5: Add Source Attribution

**Plan**: Update your question-answering component to display the source context alongside the generated answer in its output.

**Validate**: This should help you understand the basis for the information in the response.

**Execute**:

In [ ]:
# Step 5: Add Source Attribution
def rag_answer_with_sources(query, k=3):
    """RAG pipeline that returns the answer alongside its source context."""
    # Retrieve relevant context
    results = search_faq(query, k=k)
    
    # Build context from retrieved results
    context_parts = []
    for i, r in enumerate(results):
        context_parts.append(f"Q: {r['question']}\nA: {r['answer']}")
    context = "\n\n".join(context_parts)
    
    # Construct the prompt
    prompt = f"""You are a helpful research assistant. Answer the user's question based ONLY on the provided context. If the context does not contain enough information, say so.

Context:
{context}

User question: {query}

Answer:"""
    
    # Generate response
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=256, do_sample=False)
    response = output[0]["generated_text"][-1]["content"]
    
    # Display answer with source attribution
    print(f"Answer: {response}")
    print(f"\n--- Sources ---")
    for i, r in enumerate(results):
        print(f"\n  Source {i+1} (similarity: {r['score']:.4f}):")
        print(f"    Q: {r['question'][:100]}...")
        print(f"    A: {r['answer'][:100]}...")
    
    return response, results

# Test with source attribution
print("Testing RAG with source attribution:\n")
answer, sources = rag_answer_with_sources("What techniques are used for nanoparticle characterization?")

**Check**: Try asking questions. How faithful is the generated output to the retrieved context?

---

### Step 6: Add Confidence Thresholds

**Plan**: Implement a mechanism to decline answering when no strong match is found. 

**Validate**: This should output a graceful and constructive response to questions that are out of scope given the data.

**Execute**:

In [ ]:
# Step 6: Add Confidence Thresholds
CONFIDENCE_THRESHOLD = 0.35  # Minimum cosine similarity to consider a match "strong enough"

def rag_answer_with_confidence(query, k=3, threshold=CONFIDENCE_THRESHOLD):
    """RAG pipeline with confidence thresholding — declines when no strong match exists."""
    # Retrieve relevant context
    results = search_faq(query, k=k)
    
    # Check if the best match exceeds the confidence threshold
    best_score = results[0]["score"] if results else 0.0
    
    if best_score < threshold:
        decline_msg = (
            f"I don't have enough relevant information to confidently answer this question. "
            f"The best match I found had a similarity score of {best_score:.4f}, "
            f"which is below the confidence threshold of {threshold}. "
            f"This question may be outside the scope of the available FAQ data. "
            f"Consider consulting a domain expert or expanding the knowledge base."
        )
        print(f"Answer: {decline_msg}")
        print(f"\n  [Confidence: {best_score:.4f} < threshold {threshold} — DECLINED]")
        return decline_msg, results
    
    # Filter results to only those above threshold
    strong_results = [r for r in results if r["score"] >= threshold]
    
    # Build context from strong results
    context_parts = []
    for r in strong_results:
        context_parts.append(f"Q: {r['question']}\nA: {r['answer']}")
    context = "\n\n".join(context_parts)
    
    # Construct the prompt
    prompt = f"""You are a helpful research assistant. Answer the user's question based ONLY on the provided context. If the context does not contain enough information, say so.

Context:
{context}

User question: {query}

Answer:"""
    
    # Generate response
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=256, do_sample=False)
    response = output[0]["generated_text"][-1]["content"]
    
    # Display answer with source attribution
    print(f"Answer: {response}")
    print(f"\n  [Confidence: {best_score:.4f} — ACCEPTED]")
    print(f"\n--- Sources ---")
    for i, r in enumerate(strong_results):
        print(f"\n  Source {i+1} (similarity: {r['score']:.4f}):")
        print(f"    Q: {r['question'][:100]}...")
        print(f"    A: {r['answer'][:100]}...")
    
    return response, results

print(f"Confidence threshold set to {CONFIDENCE_THRESHOLD}")
print("rag_answer_with_confidence() function ready.")

**Check**: Test with both in-domain and out-of-domain questions. Does the component effectively filter responses?

In [ ]:
# Check: Test with in-domain and out-of-domain questions
print("=== IN-DOMAIN QUESTIONS ===\n")

rag_answer_with_confidence("What is the role of surfactants in nanoparticle synthesis?")

print(f"\n\n{'='*80}\n")

rag_answer_with_confidence("How does temperature affect crystal growth?")

print(f"\n\n{'='*80}")
print("=== OUT-OF-DOMAIN QUESTIONS ===\n")

rag_answer_with_confidence("What is the GDP of the United States?")

print(f"\n\n{'='*80}\n")

rag_answer_with_confidence("Who won the 2024 Super Bowl?")

**🍎 Food for thought**: How would you handle cases where your system determines incorrectly that there is a strong match in response to the input question?

### Step 7: Extend this to a set of documents

The data gave you a clean, pre-structured retrieval unit—one question, one answer. But you know that your organization has many more unstructured documents than it does manicured FAQ datasets. 

You want to explore extending this RAG pipeline you have built to documents. The `./data/` directory contains four articles and reports in `.txt` and `.md` format about AI in the enterprise that your boss had gathered for your team to read in substantiation of this AI pilot idea; they should provide a nice stalking horse for this incremental capability. 

**Plan**: Create a component to ingest the documents in the data folder and decompose them into pieces, or chunks. Load each `.txt` and `.md` file in `./data/` and split its text into overlapping chunks of a fixed size. 

**Validate**: The chunks should represent something of substance, so these should be bigger than single sentences but smaller than multiple paragraphs.

**Execute**:

In [ ]:
# Step 7: Extend to documents — chunk .txt and .md files
import os
import glob

def load_and_chunk_documents(data_dir="./data/", chunk_size=500, chunk_overlap=100):
    """
    Load .txt and .md files from data_dir, split into overlapping chunks.
    
    Args:
        data_dir: Directory containing documents
        chunk_size: Number of characters per chunk
        chunk_overlap: Number of overlapping characters between consecutive chunks
    
    Returns:
        List of dicts with 'text', 'source', and 'chunk_index' keys
    """
    chunks = []
    
    # Find all .txt and .md files
    file_patterns = [os.path.join(data_dir, "*.txt"), os.path.join(data_dir, "*.md")]
    files = []
    for pattern in file_patterns:
        files.extend(glob.glob(pattern))
    
    print(f"Found {len(files)} document(s):")
    for f in files:
        print(f"  - {os.path.basename(f)}")
    
    for filepath in files:
        filename = os.path.basename(filepath)
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
        
        # Split into overlapping chunks
        start = 0
        chunk_idx = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end].strip()
            
            if chunk_text:  # skip empty chunks
                chunks.append({
                    "text": chunk_text,
                    "source": filename,
                    "chunk_index": chunk_idx,
                })
                chunk_idx += 1
            
            start += chunk_size - chunk_overlap  # move forward with overlap
    
    print(f"\nTotal chunks created: {len(chunks)}")
    return chunks

# Create document chunks
doc_chunks = load_and_chunk_documents()

**Check**:
- Sample a few chunks. Do they seem like useful pieces of context for question answering?
- Look at where one chunk ends and the next begins. Why might you want overlap?

In [ ]:
# Check: Sample a few chunks and examine boundaries
print("=== Sample Chunks ===\n")
for i in [0, 1, 2, len(doc_chunks)//2]:
    c = doc_chunks[i]
    print(f"Chunk {i} | Source: {c['source']} | Chunk Index: {c['chunk_index']}")
    print(f"  Text ({len(c['text'])} chars): {c['text'][:200]}...")
    print()

# Show overlap between consecutive chunks from the same document
print("=== Overlap Example ===\n")
# Find two consecutive chunks from the same source
for i in range(len(doc_chunks) - 1):
    if doc_chunks[i]["source"] == doc_chunks[i+1]["source"]:
        end_of_first = doc_chunks[i]["text"][-120:]
        start_of_next = doc_chunks[i+1]["text"][:120]
        print(f"End of chunk {i}:     ...{end_of_first}")
        print(f"Start of chunk {i+1}: {start_of_next}...")
        print("\nOverlap ensures that sentences split across chunk boundaries")
        print("are still captured in at least one chunk, preserving context.")
        break

### Step 8: Build a Document Index

**Plan**: Add all the documents into a dataset that you can use for semantic similarity search (similar to your FAQ dataset above). 

**Validate**: Each entry should be a chunk of a document, the vector embedding for that chunk, and the name of the source document.

**Execute**:

In [ ]:
# Step 8: Build a Document Index
# Embed all document chunks and build a FAISS index

# Extract chunk texts for embedding
chunk_texts = [c["text"] for c in doc_chunks]

# Create embeddings for all chunks
print("Encoding document chunks...")
doc_embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)

# Normalize for cosine similarity
faiss.normalize_L2(doc_embeddings)

# Build a FAISS index for documents
doc_index = faiss.IndexFlatIP(doc_embeddings.shape[1])
doc_index.add(doc_embeddings)

print(f"\nDocument FAISS index built: {doc_index.ntotal} chunks, {doc_embeddings.shape[1]} dimensions")

# Create a DataFrame for easy access to chunk metadata
doc_df = pd.DataFrame(doc_chunks)
doc_df["embedding_dim"] = [len(e) for e in doc_embeddings]

print(f"\nDocument dataset preview:")
print(doc_df[["source", "chunk_index", "embedding_dim"]].head(10))

**Check**:
- Sample a few entries from the dataset. Do these have the data you want?
- Do the embeddings look similar to the ones representing the FAQ data?

In [ ]:
# Check: Sample entries and compare embeddings
print("=== Sample Document Index Entries ===\n")
for i in [0, 5, 10]:
    if i < len(doc_chunks):
        print(f"Entry {i}:")
        print(f"  Source:    {doc_chunks[i]['source']}")
        print(f"  Chunk #:  {doc_chunks[i]['chunk_index']}")
        print(f"  Text:     {doc_chunks[i]['text'][:100]}...")
        print(f"  Embedding shape: {doc_embeddings[i].shape}")
        print(f"  Embedding (first 5): {doc_embeddings[i][:5]}")
        print()

# Compare embedding dimensions between FAQ and document datasets
print("=== Embedding Comparison ===")
print(f"FAQ embedding dimension:      {question_embeddings.shape[1]}")
print(f"Document embedding dimension: {doc_embeddings.shape[1]}")
print(f"Same model, same dimension — embeddings are directly comparable.")

### Step 9: Query and Compare

**Plan**: Create a new RAG pipeline for this dataset using the same components you had built earlier to allow for the same question answering faculties (i.e., using similarity, source attribution and confidence thresholds) against these data as you had for the FAQs.

**Validate**: This should allow you to ask questions as input and receive grounded responses as output, just like earlier.

**Execute**:

In [ ]:
# Step 9: Query and Compare — Document RAG pipeline

def search_documents(query, k=5):
    """Search the document index for chunks similar to the query."""
    query_embedding = embedding_model.encode([query])
    faiss.normalize_L2(query_embedding)
    scores, indices = doc_index.search(query_embedding, k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "text": doc_chunks[idx]["text"],
            "source": doc_chunks[idx]["source"],
            "chunk_index": doc_chunks[idx]["chunk_index"],
        })
    return results

def doc_rag_answer(query, k=3, threshold=CONFIDENCE_THRESHOLD):
    """RAG pipeline for documents with source attribution and confidence thresholding."""
    results = search_documents(query, k=k)
    
    best_score = results[0]["score"] if results else 0.0
    
    if best_score < threshold:
        decline_msg = (
            f"I don't have enough relevant information in the document corpus to answer this question. "
            f"Best match score: {best_score:.4f} (threshold: {threshold}). "
            f"This question may be outside the scope of the available documents."
        )
        print(f"Answer: {decline_msg}")
        print(f"\n  [Confidence: {best_score:.4f} < threshold {threshold} — DECLINED]")
        return decline_msg, results
    
    strong_results = [r for r in results if r["score"] >= threshold]
    
    context_parts = []
    for r in strong_results:
        context_parts.append(f"[From: {r['source']}]\n{r['text']}")
    context = "\n\n".join(context_parts)
    
    prompt = f"""You are a helpful research assistant. Answer the user's question based ONLY on the provided context from documents. If the context does not contain enough information, say so.

Context:
{context}

User question: {query}

Answer:"""
    
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=256, do_sample=False)
    response = output[0]["generated_text"][-1]["content"]
    
    print(f"Answer: {response}")
    print(f"\n  [Confidence: {best_score:.4f} — ACCEPTED]")
    print(f"\n--- Sources ---")
    for i, r in enumerate(strong_results):
        print(f"\n  Source {i+1} (similarity: {r['score']:.4f}) — {r['source']}:")
        print(f"    {r['text'][:120]}...")
    
    return response, results

print("doc_rag_answer() function ready.")

**Check**: Try a few questions. How well does this new RAG pipeline work? 
- Try asking the same question to this RAG pipeline and your first RAG pipeline. Do they provide the same response?

In [ ]:
# Check: Test the document RAG pipeline and compare with the FAQ pipeline
shared_query = "How is AI being adopted in enterprise settings?"

print("=" * 80)
print(f"QUERY: {shared_query}")
print("=" * 80)

print("\n--- Document RAG Pipeline ---\n")
doc_answer, doc_sources = doc_rag_answer(shared_query)

print(f"\n\n{'=' * 80}")
print("\n--- FAQ RAG Pipeline (same query) ---\n")
faq_answer, faq_sources = rag_answer_with_confidence(shared_query)

print(f"\n\n{'=' * 80}")
print("COMPARISON:")
print("The document pipeline retrieves from enterprise AI articles,")
print("while the FAQ pipeline retrieves from ChemLit-QA scientific FAQs.")
print("The same query yields different (and appropriately different) results")
print("because the underlying knowledge bases cover different domains.")

# Test another query — one more in-domain for documents
print(f"\n\n{'=' * 80}")
print("\n--- Additional Document Query ---\n")
doc_rag_answer("What are the main challenges companies face when deploying AI?")

# Test an out-of-domain query for documents
print(f"\n\n{'=' * 80}")
print("\n--- Out-of-domain for Documents ---\n")
doc_rag_answer("What is the melting point of sodium chloride?")

**🍎 Food for thought**: How do you evaluate a system like this? How do you evaluate whether you made the right choices in processing, representation, etc.?

---

## End of Act II

Before proceeding to Act 3, document your work by creating a `README.md` file:

### README.md

Create a `README.md` file in your working directory that includes:

1. **Solution Summary**: A brief, plain-language explanation of what your RAG prototype does and how it works—written for a smart colleague who hasn't seen the code.

2. **Limitations and Assumptions**: Any known constraints, edge cases, or assumptions your solution relies on (e.g., FAQ-structured data, threshold tuning needs, domain limitations).

3. **Suggested Next Steps**: Concrete recommendations for what would need to happen to move from prototype to production—focusing on scaling to full documents based on your experimentation at the end, evaluation infrastructure, and user interface design.

---

### Act 3: Socialize the solution with stakeholders

## Socializing with Stakeholders

You've built a prototype RAG system for cross-team knowledge access. Now imagine presenting it to three colleagues at B10 who would be affected by deploying this technology. Use GitHub Copilot in **Ask** mode to roleplay as each colleague.

---

### Colleague 1: Senior Research Scientist

This colleague leads a specialized research team and is protective of their team's knowledge and methodologies. They're concerned about how their expertise might be shared or misrepresented.

**Prompt Copilot to respond as this stakeholder:**
- What concerns would they have about making their team's knowledge accessible to others?
- How might they want to control or approve what gets surfaced?
- What would make them trust this system?

---

### Colleague 2: IT Security & Compliance Lead

This colleague ensures data security and regulatory compliance. They're particularly concerned about how sensitive research data is handled and accessed.

**Prompt Copilot to respond as this stakeholder:**
- What security concerns might they raise about the RAG architecture?
- How would they want to audit or log system usage?
- What compliance requirements (e.g., FDA, IP protection) would need to be addressed?

---

### Colleague 3: VP of Research Operations

This colleague is responsible for research efficiency and cross-team collaboration. They're excited about the potential but need to understand the business case and risks.

**Prompt Copilot to respond as this stakeholder:**
- What ROI would they need to see to justify deployment?
- How would they measure success?
- What governance structure would they want around the system?

---

**Document each conversation** by saving it or exporting it from GitHub Copilot. These perspectives will help you anticipate real-world concerns when presenting AI solutions to business stakeholders.

---

### Submission

1. **Save your files**: Ensure your notebook (`.ipynb`), `agents.md`, and `README.md` are all saved.

2. **Export your chat history**: Save or export your GitHub Copilot chat conversations, including the stakeholder roleplay sessions.

3. **Stop your session**: Shut down your or VS Code kernel.

4. **Upload to Canvas**: Submit your notebook and supporting files to [Canvas](https://canvas.northwestern.edu/courses/245397/assignments/1668986).